# Document Question Answering System using RAG

A **Retrieval-Augmented Generation (RAG)** pipeline built on the [vectara/open_ragbench](https://huggingface.co/datasets/vectara/open_ragbench) dataset.

**Pipeline:**
1. Load documents from the HuggingFace dataset
2. Chunk text into retrievable segments
3. Generate semantic embeddings for each chunk
4. Store embeddings in a FAISS vector database
5. Retrieve relevant chunks for a user query
6. Generate a grounded answer using an LLM


## 1. Install Required Libraries

In [1]:
!pip install datasets faiss-cpu sentence-transformers transformers torch -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 64.4 MB/s eta 0:00:00


## 2. Import Libraries

In [2]:
import warnings
warnings.filterwarnings("ignore")

import textwrap
import numpy as np
import faiss

from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from transformers import pipeline

## 3. Load the HuggingFace Dataset

`vectara/open_ragbench` contains documents with associated questions and ground-truth answers across multiple domains. We use the `covidqa` subset.


In [4]:
dataset = load_dataset("rungalileo/ragbench", "covidqa", split="test")

print(f"Rows loaded  : {len(dataset)}")
print(f"Columns      : {dataset.column_names}")

README.md:   0%|          | 0.00/24.7k [00:00<?, ?B/s]

covidqa/train-00000-of-00001.parquet:   0%|          | 0.00/4.20M [00:00<?, ?B/s]

covidqa/test-00000-of-00001.parquet:   0%|          | 0.00/854k [00:00<?, ?B/s]

covidqa/validation-00000-of-00001.parque(…):   0%|          | 0.00/913k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1252 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/246 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/267 [00:00<?, ? examples/s]

Rows loaded  : 246
Columns      : ['id', 'question', 'documents', 'response', 'generation_model_name', 'annotating_model_name', 'dataset_name', 'documents_sentences', 'response_sentences', 'sentence_support_information', 'unsupported_response_sentence_keys', 'adherence_score', 'overall_supported_explanation', 'relevance_explanation', 'all_relevant_sentence_keys', 'all_utilized_sentence_keys', 'trulens_groundedness', 'trulens_context_relevance', 'ragas_faithfulness', 'ragas_context_relevance', 'gpt3_adherence', 'gpt3_context_relevance', 'gpt35_utilization', 'relevance_score', 'utilization_score', 'completeness_score']


## 4. Explore a Sample Entry

In [9]:
sample = dataset[0]
print(sample)

print(f"Question : {sample['question']}")
print(f"Answer   : {sample['response']}")
print(f"\nContext (first 400 chars):")
print(sample['documents'][0][:400])

{'id': '1421', 'question': 'Which viruses may not cause prolonged inflammation due to strong induction of antiviral clearance?', 'documents': ['Title: Type I Interferon Receptor Deficiency in Dendritic Cells Facilitates Systemic Murine Norovirus Persistence Despite Enhanced Adaptive Immunity\nPassage: successful treatment for HCV serves to circumvent the viral inhibition of IFN induction. Thus, HCV may be an example of a medically relevant persistent viral infection that persists due, in part, to loss of innate immune function. Persistence of other continuously replicating RNA viruses, such as chikungunya, measles, polyomavirus, may be similarly due to ineffective innate responses.', 'Title: Type I Interferon Response Is Delayed in Human Astrovirus Infections\nPassage: Results suggest that HAstV infection is not able to disrupt the innate immune sensing pathway induced by polyI:C . Only a previous infection with RV was able to reduce by 60% the IFN-β mRNA levels produced after polyI:C 

## 5. Build the Document Corpus

We extract unique context passages from the dataset. These passages are the raw documents the RAG system will retrieve from.


In [11]:
corpus = list(set([doc for row in dataset for doc in row["documents"] if doc.strip()]))

print(f"Unique passages in corpus: {len(corpus)}")
print(f"\nSample passage (first 300 chars):")
print(corpus[0][:300])

Unique passages in corpus: 902

Sample passage (first 300 chars):
Title: Influenza A viruses are transmitted via the air from the nasal respiratory epithelium of ferrets
Passage: M illions of people have lost their lives due to influenza A virus epidemics and pandemics. Prevention and control of IAV infections are based on vaccination and treatment. However, a bet


## 6. Text Chunking

Each passage is split into smaller overlapping chunks. Smaller chunks improve retrieval precision — the model can pinpoint the exact paragraph that answers a question rather than returning a full document.


In [12]:
def chunk_text(text, chunk_size=300, overlap=50):
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        end = min(start + chunk_size, len(words))
        chunks.append(" ".join(words[start:end]))
        start += chunk_size - overlap
    return chunks

all_chunks = []
for passage in corpus:
    all_chunks.extend(chunk_text(passage))

print(f"Total chunks created     : {len(all_chunks)}")
print(f"Avg chunk length (words) : {np.mean([len(c.split()) for c in all_chunks]):.0f}")

Total chunks created     : 902
Avg chunk length (words) : 88


## 7. Generate Embeddings

Each chunk is converted to a dense vector using `all-MiniLM-L6-v2` — a fast, CPU-friendly sentence-transformer that captures semantic meaning.


In [13]:
embedder = SentenceTransformer("all-MiniLM-L6-v2")

print("Encoding all chunks...")
chunk_embeddings = embedder.encode(
    all_chunks,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True
)

print(f"\nEmbedding matrix shape: {chunk_embeddings.shape}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Encoding all chunks...


Batches:   0%|          | 0/15 [00:00<?, ?it/s]


Embedding matrix shape: (902, 384)


## 8. Store Embeddings in FAISS Vector Database

FAISS (Facebook AI Similarity Search) stores the embeddings and supports fast nearest-neighbour lookup. We use an inner-product index after L2 normalisation, which is equivalent to cosine similarity search.


In [17]:
dimension = chunk_embeddings.shape[1]

faiss.normalize_L2(chunk_embeddings)

vector_store = faiss.IndexFlatIP(dimension)
vector_store.add(chunk_embeddings)

print(f"Vector store ready: {vector_store.ntotal} vectors, dimension {dimension}")

Vector store ready: 902 vectors, dimension 384


## 9. Retrieve Relevant Chunks

Given a user query, we embed it with the same model and search the vector store for the most semantically similar chunks.


In [15]:
def retrieve(query, top_k=5):
    query_vec = embedder.encode([query], convert_to_numpy=True)
    faiss.normalize_L2(query_vec)
    distances, indices = vector_store.search(query_vec, top_k)

    results = []
    for rank, (score, idx) in enumerate(zip(distances[0], indices[0]), 1):
        results.append({
            "rank": rank,
            "score": round(float(score), 4),
            "chunk": all_chunks[idx]
        })
    return results

query = "What are the symptoms of COVID-19?"
retrieved = retrieve(query)

print(f"Query: {query}\n")
for r in retrieved:
    print(f"Rank {r['rank']}  |  Score: {r['score']}")
    print(textwrap.fill(r['chunk'][:250], width=90))
    print()

Query: What are the symptoms of COVID-19?

Rank 1  |  Score: 0.5881
Title: CDC Summary 21 MAR 2020, Passage: People who get a fever or cough should consider
whether they might have COVID-19, depending on where they live, their travel history or
other exposures. More than half of the U.S. is seeing some level of commu

Rank 2  |  Score: 0.5721
Title: CDC Summary 21 MAR 2020, Passage: Early information out of China, where COVID-19
first started, shows that some people are at higher risk of getting very sick from this
illness. This includes:

Rank 3  |  Score: 0.5361
Title: Species‐specific clinical characteristics of human coronavirus infection among
otherwise healthy adolescents and adults Passage: DOI: 10.1111/irv.12538

Rank 4  |  Score: 0.5336
Title: Species‐specific clinical characteristics of human coronavirus infection among
otherwise healthy adolescents and adults Passage:
https://www.ncbi.nlm.nih.gov/pmc/articles/PMC5820427/

Rank 5  |  Score: 0.5214
Title: Species‐specific clin

## 10. Load the Answer Generation Model

We use `google/flan-t5-base` — a lightweight instruction-tuned model that runs on CPU. It takes the retrieved context as input and generates a grounded answer.


In [28]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

# Load the tokenizer and model directly
tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")

# Define a custom generator function to mimic the pipeline output
def custom_generator(prompt, max_new_tokens=200, return_full_text=False):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)

    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # The previous rag_answer logic expects a list of dicts with 'generated_text'
    return [{
        "generated_text": generated_text.strip()
    }]

generator = custom_generator

print("Answer generation model loaded and custom generator defined.")

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Answer generation model loaded and custom generator defined.


## 11. Full RAG Pipeline

This is the complete pipeline: retrieve relevant chunks → build a prompt with context → generate an answer.


In [29]:
def rag_answer(question, top_k=5):
    retrieved_chunks = retrieve(question, top_k=top_k)
    context = "\n\n".join([r["chunk"] for r in retrieved_chunks])

    prompt = (
        "Answer the following question using only the provided context.\n\n"
        f"Context:\n{context[:1500]}\n\n"
        f"Question: {question}\n\n"
        "Answer:"
    )

    # Generate text. Some models, even with return_full_text=False, might echo the prompt.
    raw_generated_text = generator(prompt, max_new_tokens=200, return_full_text=False)[0]["generated_text"].strip()

    # Robustly extract the answer by removing the prompt from the generated text.
    # Using replace with count=1 ensures only the first occurrence (the echoed prompt) is removed.
    answer = raw_generated_text.replace(prompt, "", 1).strip()

    # If the model didn't generate an answer after the prompt, return a placeholder
    if not answer:
        answer = "No answer generated based on the provided context."

    return {
        "question": question,
        "answer": answer,
        "retrieved_chunks": retrieved_chunks
    }

## 12. Ask Questions — End to End

In [30]:
questions = [
    "What are the symptoms of COVID-19?",
    "How does the coronavirus spread from person to person?",
    "What is the incubation period of COVID-19?",
    "What precautions help prevent COVID-19?",
]

for question in questions:
    result = rag_answer(question)
    print(f"Question : {result['question']}")
    print(f"Answer   : {result['answer']}")
    print("-" * 80)

Question : What are the symptoms of COVID-19?
Answer   : fever or cough
--------------------------------------------------------------------------------
Question : How does the coronavirus spread from person to person?
Answer   : from infected to uninfected humans in close and prolonged contact through circumstances created by poor infection control in health care settings
--------------------------------------------------------------------------------
Question : What is the incubation period of COVID-19?
Answer   : two to 16 days
--------------------------------------------------------------------------------
Question : What precautions help prevent COVID-19?
Answer   : People who get a fever or cough should consider whether they might have COVID-19
--------------------------------------------------------------------------------


## 13. Inspect Retrieved Context for Any Answer

This helper shows both the generated answer and the exact chunks that grounded it — making the system fully transparent and auditable.


In [31]:
def ask(question, top_k=5):
    result = rag_answer(question, top_k=top_k)

    print(f"Question: {result['question']}")
    print(f"\nAnswer: {result['answer']}")
    print("\nRetrieved context chunks:")
    for r in result["retrieved_chunks"]:
        print(f"  Rank {r['rank']}  |  Score {r['score']}")
        print(f"  {r['chunk'][:220]}...")
        print()

ask("Are children more susceptible to COVID-19 than adults?")

Question: Are children more susceptible to COVID-19 than adults?

Answer: Children are at higher risk of COVID-19 infection than adults.

Retrieved context chunks:
  Rank 1  |  Score 0.6531
  Title: CDC Summary 21 MAR 2020, Passage: The risk from COVID-19 to Americans can be broken down into risk of exposure versus risk of serious illness and death....

  Rank 2  |  Score 0.6291
  Title: CDC Summary 21 MAR 2020, Passage: Early information out of China, where COVID-19 first started, shows that some people are at higher risk of getting very sick from this illness. This includes:...

  Rank 3  |  Score 0.6269
  Title: CDC Summary 21 MAR 2020, Passage: People who get a fever or cough should consider whether they might have COVID-19, depending on where they live, their travel history or other exposures. More than half of the U.S....

  Rank 4  |  Score 0.6163
  Title: Species‐specific clinical characteristics of human coronavirus infection among otherwise healthy adolescents and adults Pass